# Módulo 03 · Aula 4: Avaliação da recuperação

**Duração:** 45 minutos

RAG sem avaliação é opinião. Até aqui, testamos o assistente com perguntas soltas e dissemos "parece bom". Nas
últimas aulas do curso trocamos a impressão por números: primeiro medimos a **busca** (esta aula), depois a
**resposta**, a **segurança** e o **custo** (aula 3.5) e, no projeto final (módulo 04), colocamos o assistente no ar
com documentos novos.

## Objetivos de aprendizagem

- entender o conjunto de avaliação `data/eval/questions.json`: tipos de pergunta, fontes esperadas e por que as
  perguntas `no_answer` ficam fora das métricas de busca;
- calcular **hit@k**, **reciprocal rank** (e a média, **MRR**) e **coverage**, à mão e com código;
- comparar configurações de busca com números: vetorial x BM25 x híbrida, `k` = 2, 4 e 8, com e sem reordenação;
- analisar os erros pergunta a pergunta e separar "a busca errou" de "o gabarito está incompleto".

## Pré-requisitos

Módulo 02: `HybridRetriever` com os modos `vector`, `bm25` e `hybrid`, a fusão RRF e a reordenação com o LLM
(`rerank_with_llm`). O índice principal `data/faiss_index` (criado no módulo 01).

## O que vamos construir

Uma bancada de testes da busca: a mesma lista de perguntas passa por várias configurações, e cada configuração
recebe uma nota.

```mermaid
flowchart LR
    Q[(questions.json<br/>30 perguntas)] --> F[26 com fonte esperada]
    F --> S["search(pergunta)<br/>vector · bm25 · hybrid · k"]
    S --> R[fontes recuperadas]
    R --> M[retrieval_metrics<br/>hit · rr · coverage]
    M --> T[tabela por pergunta]
    T --> A[resumo<br/>hit rate · MRR · coverage]
```

> ⚠️ Esta aula chama a API de embeddings uma vez por pergunta (usamos um cache para não repetir) e, na seção 7,
> o LLM em 10 perguntas. Os números de embeddings são estáveis; os da reordenação podem variar um pouco.

In [1]:
# Os notebooks rodam a partir da raiz do projeto: é lá que ficam o .env, a pasta data/ e o pacote src/.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Chat e embeddings vêm do Azure OpenAI configurado no .env (veja src/config.py).
from src.config import settings

## 1. O conjunto de avaliação

Um conjunto de avaliação é uma lista **fixa** de perguntas com o que se espera de cada uma. Fixa é a palavra
importante: se a lista muda a cada teste, os números deixam de ser comparáveis. O nosso tem 30 perguntas, escritas
como um cliente escreveria (com "pra", "tá", erros de digitação e contexto pessoal), e cada uma traz:

| Chave | Conteúdo |
|---|---|
| `id` | identificador estável (`q01`...`q30`) |
| `type` | `fact`, `procedure`, `multi_document` ou `no_answer` |
| `question` | a pergunta do cliente |
| `reference_answer` | a resposta correta, escrita por quem conhece os documentos (usada na aula 3.5) |
| `expected_sources` | os arquivos que contêm a resposta (usados nesta aula) |

`load_questions()` lê o arquivo:

In [2]:
import pandas as pd

from src.evaluation import load_questions

pd.set_option("display.max_colwidth", 90)

questions = load_questions()
questions_by_id = {q["id"]: q for q in questions}
eval_df = pd.DataFrame(questions)
print(f"{len(questions)} perguntas")
eval_df["type"].value_counts()

30 perguntas


type
fact              12
procedure          8
multi_document     6
no_answer          4
Name: count, dtype: int64

In [3]:
eval_df.groupby("type").head(1)[["id", "type", "question", "expected_sources"]]

,id,type,question,expected_sources
0,q01,fact,Qual o valor mínimo da compra pra ter frete grátis?,[policies/shipping.md]
12,q13,procedure,Como faço pra rastrear meu pedido?,[help_center/track_order.html]
20,q21,multi_document,Comprei um Notebook Nova Air 14 há 40 dias e ele parou de ligar. Ainda consigo meu din...,"[policies/refund.md, policies/warranty.md]"
26,q27,no_answer,Vocês vendem garantia estendida? Quanto custa?,[]


Cada tipo testa uma habilidade diferente:

| Tipo | O que testa | Exemplo |
|---|---|---|
| `fact` | achar um dado pontual (valor, prazo, especificação) | valor mínimo para frete grátis |
| `procedure` | achar um passo a passo | como rastrear o pedido |
| `multi_document` | juntar duas fontes (política + manual, pagamento + frete) | notebook com defeito após 40 dias |
| `no_answer` | admitir que a base não tem a resposta | placa de vídeo de um notebook que não existe no acervo |

Duas decisões de projeto merecem atenção:

- **As fontes esperadas são arquivos, não trechos.** Se amanhã mudarmos o `chunk_size`, os ids dos trechos mudam,
  mas `policies/shipping.md` continua sendo a fonte certa. O gabarito sobrevive às mudanças de ingestão.
- **As perguntas `no_answer` não têm fonte esperada.** Não existe "trecho certo" para encontrar: qualquer coisa que
  a busca devolva é, por definição, irrelevante. Essas perguntas avaliam a **geração** (o assistente admite que não
  sabe?), e por isso ficam para a aula 3.5.

In [4]:
with_sources = [q for q in questions if q["expected_sources"]]
print(f"Com fonte esperada: {len(with_sources)} | no_answer (fora das métricas de busca): {len(questions) - len(with_sources)}")
for q in questions:
    if q["type"] == "no_answer":
        print(f"- {q['id']}: {q['question']} -> expected_sources={q['expected_sources']}")

Com fonte esperada: 26 | no_answer (fora das métricas de busca): 4
- q27: Vocês vendem garantia estendida? Quanto custa? -> expected_sources=[]
- q28: Qual é o modelo da placa de vídeo do Notebook Nova Gamer 16? -> expected_sources=[]
- q29: A TechNova está contratando? Onde mando meu currículo? -> expected_sources=[]
- q30: Quando vai ser lançado o Smartphone Nova X 2? -> expected_sources=[]


## 2. As métricas: hit@k, reciprocal rank e coverage

Para cada pergunta, a busca devolve uma lista ordenada com `k` trechos. Olhamos a **fonte** (`metadata["source"]`)
de cada um e comparamos com as fontes esperadas:

| Métrica | Fórmula (por pergunta) | Pergunta que responde |
|---|---|---|
| **hit@k** | 1 se alguma fonte esperada aparece nos `k` trechos; 0 se não | o trecho certo chegou ao modelo? |
| **reciprocal rank (rr)** | 1 / posição da primeira fonte esperada (0 se não aparece) | chegou em que posição? |
| **coverage** | fontes esperadas encontradas / fontes esperadas | chegaram **todas** as fontes necessárias? |

A média de cada uma no conjunto dá o **hit rate**, o **MRR** (Mean Reciprocal Rank) e a **coverage média**.

### Um exemplo calculado à mão

Pergunta `q23`: *"Minha Nova Boom caiu na piscina com a tampinha do USB aberta e parou de funcionar. A garantia
cobre?"*. Fontes esperadas: o manual `manuals/cx-001_nova_boom.pdf` **e** a política `policies/warranty.md`.
Suponha que a busca devolveu, com `k = 4`:

| Posição | Fonte recuperada | É esperada? |
|---|---|---|
| 1 | `faq.csv` | não |
| 2 | `manuals/cx-001_nova_boom.pdf` | **sim** |
| 3 | `help_center/warranty_claim_step_by_step.html` | não |
| 4 | `manuals/cx-001_nova_boom.pdf` | sim (repetida) |

- **hit@4** = 1: uma fonte esperada apareceu.
- **rr** = 1/2 = **0,5**: a primeira fonte esperada está na posição 2.
- **coverage** = 1/2 = **0,5**: o manual apareceu, a política de garantia não. O modelo vai saber que a IPX7 exige a
  tampa fechada, mas não vai ter a regra da garantia sobre mau uso.

E o MRR de três perguntas com rr = 1, 0,5 e 0 é (1 + 0,5 + 0) / 3 = **0,5**.

Vamos escrever as três funções do zero para conferir:

In [5]:
def hit_at_k(retrieved: list[str], expected: list[str]) -> int:
    return int(any(source in expected for source in retrieved))


def reciprocal_rank(retrieved: list[str], expected: list[str]) -> float:
    for position, source in enumerate(retrieved, start=1):
        if source in expected:
            return 1 / position
    return 0.0


def coverage(retrieved: list[str], expected: list[str]) -> float:
    return len(set(expected) & set(retrieved)) / len(set(expected))


example_retrieved = [
    "faq.csv",
    "manuals/cx-001_nova_boom.pdf",
    "help_center/warranty_claim_step_by_step.html",
    "manuals/cx-001_nova_boom.pdf",
]
example_expected = questions_by_id["q23"]["expected_sources"]

print("hit@4   :", hit_at_k(example_retrieved, example_expected))
print("rr      :", reciprocal_rank(example_retrieved, example_expected))
print("coverage:", coverage(example_retrieved, example_expected))
print("MRR de três perguntas (1, 0,5 e 0):", sum([1, 0.5, 0]) / 3)

hit@4   : 1
rr      : 0.5
coverage: 0.5
MRR de três perguntas (1, 0,5 e 0): 0.5


O produto tem a mesma lógica em uma única função, `retrieval_metrics`, em
[src/evaluation.py](../src/evaluation.py). Repare que nenhuma das funções recebe `k`: **o @k está em quem busca**.
A métrica mede a lista que recebeu; se a busca devolveu 4 trechos, é hit@4.

In [6]:
from src.evaluation import retrieval_metrics

metrics = retrieval_metrics(example_retrieved, example_expected)
print(metrics)
assert metrics == {
    "hit": hit_at_k(example_retrieved, example_expected),
    "rr": reciprocal_rank(example_retrieved, example_expected),
    "coverage": coverage(example_retrieved, example_expected),
}
print("As três funções do notebook batem com retrieval_metrics.")

{'hit': 1, 'rr': 0.5, 'coverage': 0.5}
As três funções do notebook batem com retrieval_metrics.


Qual métrica olhar primeiro?

- **hit rate** é a mais importante em RAG: o modelo lê **todos** os `k` trechos, então o que importa é o trecho
  certo estar na lista. Um erro aqui não tem conserto no prompt.
- **MRR** importa quando `k` é pequeno, quando reordenamos, e porque trechos no topo tendem a pesar mais na
  resposta. Um MRR baixo com hit rate alto diz "achou, mas enterrou".
- **coverage** só difere do hit nas perguntas `multi_document`, que precisam de duas fontes para uma resposta
  completa.

> 💡 **Dica:** métricas de busca são **determinísticas** e baratas (sem LLM). Por isso vale rodá-las a cada mudança
> na ingestão ou na busca, como um teste automatizado.

## 3. Preparando a medição: um cache de embeddings de consulta

Vamos rodar as mesmas 26 perguntas em várias configurações. As buscas vetorial e híbrida calculam o embedding da
pergunta a **cada** chamada: 26 perguntas x 6 configurações com vetor = 156 chamadas à API para os mesmos 26
vetores. Um cache simples resolve: uma subclasse de `Embeddings` que guarda o vetor de cada consulta já vista e
repassa o resto para o modelo de verdade.

Carregamos o índice com esse objeto (`load_index(embeddings=...)`): é o mesmo modelo que criou o índice, só que com
memória.

In [7]:
from langchain_core.embeddings import Embeddings

from src.models import get_embeddings
from src.rag.retrieval import HybridRetriever
from src.rag.vectorstore import index_exists, load_index


class CachedQueryEmbeddings(Embeddings):
    """Reaproveita o vetor de consultas repetidas; os documentos passam direto para o modelo."""

    def __init__(self, base: Embeddings):
        self.base = base
        self.cache: dict[str, list[float]] = {}
        self.api_calls = 0

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.base.embed_documents(texts)

    def embed_query(self, text: str) -> list[float]:
        if text not in self.cache:
            self.cache[text] = self.base.embed_query(text)
            self.api_calls += 1
        return self.cache[text]


if not index_exists():
    from src.rag.ingestion import index_folder

    index_folder()

embeddings = CachedQueryEmbeddings(get_embeddings())
index = load_index(embeddings=embeddings)
retriever = HybridRetriever(index=index)
print(f"{index.index.ntotal} trechos | modo padrão: {retriever.mode} | k padrão: {retriever.k} | fetch_k: {retriever.fetch_k}")

2026-09-27 22:17:18 | INFO     | src.models | Inicializando embeddings | provedor=azure | deployment=text-embedding-3-large


321 trechos | modo padrão: hybrid | k padrão: 4 | fetch_k: 12


## 4. Primeira medição: a configuração do produto

`evaluate_retrieval(search, questions=None)` recebe uma **função de busca**: qualquer coisa que transforme uma
pergunta em uma lista de `Document`. Assim ela avalia qualquer retriever, qualquer configuração e até um serviço
externo. Com uma `lambda` fixamos os parâmetros de cada experimento. Ela devolve a tabela por pergunta e o resumo.

Começamos pela configuração padrão do produto: busca híbrida com `k = 4`.

In [8]:
from src.evaluation import evaluate_retrieval

table, summary = evaluate_retrieval(lambda q: retriever.search(q, k=4, mode="hybrid"))
print(summary)
table[["id", "type", "hit", "rr", "coverage"]].head(8)

{'questions': 26, 'hit_rate': 0.923, 'mrr': 0.683, 'coverage': 0.904}


,id,type,hit,rr,coverage
0,q01,fact,1,0.5,1.0
1,q02,fact,0,0.0,0.0
2,q03,fact,1,1.0,1.0
3,q04,fact,1,0.5,1.0
4,q05,fact,1,0.5,1.0
5,q06,fact,1,1.0,1.0
6,q07,fact,1,1.0,1.0
7,q08,fact,1,1.0,1.0


In [9]:
table.groupby("type")[["hit", "rr", "coverage"]].mean().round(3)

,hit,rr,coverage
type,,,
fact,0.917,0.688,0.917
multi_document,0.833,0.611,0.750
procedure,1.000,0.729,1.000


**Observe:** as perguntas `multi_document` têm a coverage mais baixa. Achar **uma** das fontes é fácil; achar as
duas, com só 4 vagas, é o desafio. É o primeiro candidato a melhoria.

## 5. Comparando configurações

Agora a grade completa: três modos x três valores de `k`. Como `fetch_k = 12` candidatos por busca e o `k` só corta
a lista final, o custo extra é só o processamento local: os vetores das perguntas já estão no cache.

> 💡 **Dica:** a `lambda` é usada na hora, dentro do laço, então enxerga o `mode` e o `k` daquela volta. Se você
> guardasse as lambdas em uma lista para usar depois, todas veriam os **últimos** valores (as closures do Python
> leem a variável no momento da chamada, não no da criação).

In [10]:
rows, tables = [], {}
for mode in ("vector", "bm25", "hybrid"):
    for k in (2, 4, 8):
        t, s = evaluate_retrieval(lambda q: retriever.search(q, k=k, mode=mode))
        tables[(mode, k)] = t
        rows.append({"mode": mode, "k": k, **s})

comparison = pd.DataFrame(rows).set_index(["mode", "k"])
print(f"Chamadas à API de embeddings até aqui: {embeddings.api_calls} (uma por pergunta)")
comparison

Chamadas à API de embeddings até aqui: 26 (uma por pergunta)


questions  hit_rate    mrr  coverage
mode   k                                      
vector 2         26     0.885  0.731     0.808
       4         26     0.962  0.756     0.904
       8         26     0.962  0.756     0.942
bm25   2         26     0.615  0.519     0.577
       4         26     0.885  0.603     0.827
       8         26     0.962  0.615     0.923
hybrid 2         26     0.769  0.635     0.731
       4         26     0.923  0.683     0.904
       8         26     0.962  0.690     0.962

In [11]:
comparison.reset_index().pivot(index="mode", columns="k", values=["hit_rate", "mrr", "coverage"])

hit_rate                  mrr               coverage              
k             2      4      8      2      4      8        2      4      8
mode                                                                     
bm25      0.615  0.885  0.962  0.519  0.603  0.615    0.577  0.827  0.923
hybrid    0.769  0.923  0.962  0.635  0.683  0.690    0.731  0.904  0.962
vector    0.885  0.962  0.962  0.731  0.756  0.756    0.808  0.904  0.942

**Observe** (com `k = 4`, referência do produto: vetorial 0,962 / MRR 0,756; BM25 0,885 / 0,603; híbrida
0,923 / 0,683):

- **`k` maior aumenta hit rate e coverage**, mas quase não mexe no MRR: o MRR só muda quando a primeira fonte
  certa estava **depois** do corte. Mais trechos também significam mais tokens por pergunta (aula 3.5).
- **A vetorial vence nesta base**, e a híbrida fica entre as duas. As perguntas foram escritas em linguagem natural,
  com paráfrases ("pessoa de verdade no sábado" x "atendentes humanos"); o BM25 depende das mesmas palavras e puxa a
  fusão para baixo. A híbrida brilha com códigos e nomes exatos (`FN-003`, `IPX7`, `NOVA50`), e o conjunto tem
  poucas perguntas assim.
- **O conjunto decide o vencedor.** Se os clientes de verdade digitam códigos de produto, o conjunto precisa ter
  perguntas assim; senão, a avaliação otimiza para um cliente que não existe.
- **Cuidado com diferenças pequenas.** Com 26 perguntas, cada pergunta vale 3,8 pontos de hit rate: vetorial
  (0,962) e híbrida (0,923) diferem em **uma** pergunta. Antes de trocar a configuração do produto, olhe **quais**
  perguntas mudaram e por quê.

## 6. Análise dos erros

Um número médio diz **quanto** erramos; a tabela por pergunta diz **onde** e sugere **por quê**. Vamos olhar as
perguntas em que a configuração do produto (híbrida, `k = 4`) não colocou a fonte esperada em primeiro lugar:

In [12]:
product_table = tables[("hybrid", 4)]
misses = product_table[product_table["rr"] < 1][["id", "type", "rr", "coverage", "retrieved", "expected"]]
print(f"{len(misses)} de {len(product_table)} perguntas com rr < 1 | sem nenhuma fonte esperada: {int((misses['rr'] == 0).sum())}")
misses

13 de 26 perguntas com rr < 1 | sem nenhuma fonte esperada: 2


,id,type,rr,coverage,retrieved,expected
0,q01,fact,0.500000,1.0,"[faq.csv, policies/shipping.md, help_center/coupon_not_applied.html, help_center/track...",[policies/shipping.md]
1,q02,fact,0.000000,0.0,"[help_center/warranty_claim_step_by_step.html, policies/exchange.md, manuals/nb-001_no...",[policies/warranty.md]
3,q04,fact,0.500000,1.0,"[faq.csv, manuals/mn-001_nova_view_27.pdf, manuals/mn-001_nova_view_27.pdf, manuals/mn...",[manuals/mn-001_nova_view_27.pdf]
4,q05,fact,0.500000,1.0,"[faq.csv, manuals/fn-001_nova_buds.pdf, manuals/fn-001_nova_buds.pdf, manuals/fn-001_n...",[manuals/fn-001_nova_buds.pdf]
8,q09,fact,0.250000,1.0,"[help_center/change_delivery_address.html, help_center/track_order.html, help_center/b...",[help_center/support_channels_and_hours.html]
11,q12,fact,0.500000,1.0,"[faq.csv, manuals/sm-001_nova_x.pdf, manuals/sm-001_nova_x.pdf, manuals/sm-001_nova_x....",[manuals/sm-001_nova_x.pdf]
14,q15,procedure,0.333333,1.0,"[faq.csv, manuals/fn-001_nova_buds.pdf, help_center/warranty_claim_step_by_step.html, ...",[help_center/warranty_claim_step_by_step.html]
16,q17,procedure,0.500000,1.0,"[faq.csv, help_center/damaged_product.html, help_center/damaged_product.html, policies...",[help_center/damaged_product.html]
18,q19,procedure,0.500000,1.0,"[faq.csv, help_center/change_delivery_address.html, policies/cancellation.md, policies...",[help_center/change_delivery_address.html]
19,q20,procedure,0.500000,1.0,"[faq.csv, help_center/reset_password.html, help_center/reset_password.html, faq.csv]",[help_center/reset_password.html]


Um padrão salta aos olhos: `faq.csv` aparece muito na posição 1. Será que a busca errou?

In [13]:
faq_first = misses[misses["retrieved"].map(lambda sources: sources[0] == "faq.csv")]
print(f"{len(faq_first)} de {len(misses)} perguntas com rr < 1 têm faq.csv na posição 1\n")

question = questions_by_id["q05"]
top = retriever.search(question["question"], k=1, mode="hybrid")[0]
print("Pergunta :", question["question"])
print("Esperado :", question["expected_sources"])
print(f"Posição 1: {top.metadata['source']}\n{top.page_content}")

9 de 13 perguntas com rr < 1 têm faq.csv na posição 1

Pergunta : Quanto dura a bateria do Fone Nova Buds com o cancelamento de ruído ligado?
Esperado : ['manuals/fn-001_nova_buds.pdf']
Posição 1: faq.csv
[Qual a autonomia da bateria do Fone Nova Buds?]
Pergunta: Qual a autonomia da bateria do Fone Nova Buds?
Resposta: Até 6 horas com o cancelamento de ruído ligado (8 horas desligado) e até 24 horas no total com o estojo, com o cancelamento de ruído ligado.


A busca **acertou**: a pergunta frequente sobre a bateria do Nova Buds responde a pergunta, só que ela não está no
gabarito. As perguntas frequentes repetem, de forma resumida, o conteúdo das políticas e dos manuais. Isso nos dá
as categorias de erro desta base:

| Categoria | Perguntas | O que fazer |
|---|---|---|
| **Gabarito incompleto**: uma fonte válida não está em `expected_sources` | `faq.csv` no topo em várias | revisar o gabarito (aceitar a linha da FAQ como fonte válida) |
| **Foco errado**: um termo da pergunta domina a busca | `q21` (só o manual do Nova Air 14) | dividir a pergunta, reordenar, deixar o grafo corretivo reescrever |
| **Documentos vizinhos**: dois arquivos sobre o mesmo tema | `q02`, `q15` (artigo de garantia x política de garantia) | reordenar; enriquecer o cabeçalho dos trechos |
| **Vocabulário diferente**: o cliente usa outras palavras | `q09` ("pessoa de verdade") | a parte vetorial ajuda; o BM25 sozinho erra |

> ⚠️ **Armadilha comum:** "consertar" o gabarito para o número subir. Só mude `expected_sources` quando a fonte
> nova **responde de fato** a pergunta (como acima), e registre a mudança: números de versões diferentes do
> gabarito não são comparáveis.

Veja o foco errado em ação na `q21`, que precisa das políticas de reembolso e de garantia:

In [14]:
question = questions_by_id["q21"]
print(question["question"], "\nEsperado:", question["expected_sources"], "\n")
for chunk in retriever.search(question["question"], k=4, mode="hybrid"):
    print(f"{chunk.metadata['source']:<34} | {chunk.page_content[:80].replace(chr(10), ' ')}")

Comprei um Notebook Nova Air 14 há 40 dias e ele parou de ligar. Ainda consigo meu dinheiro de volta? 
Esperado: ['policies/refund.md', 'policies/warranty.md'] 

manuals/nb-001_nova_air_14.pdf     | [Manual do usuário: Notebook Nova Air 14 > página 5] problema não for resolvido 
manuals/nb-001_nova_air_14.pdf     | [Manual do usuário: Notebook Nova Air 14 > página 1] TechNova Notebook Nova Air 
manuals/nb-001_nova_air_14.pdf     | [Manual do usuário: Notebook Nova Air 14 > página 5] - Danos causados por queda 
manuals/nb-001_nova_air_14.pdf     | [Manual do usuário: Notebook Nova Air 14 > página 2] - 1 Notebook Nova Air 14 - 


"Notebook Nova Air 14" é o termo mais específico da pergunta e leva os quatro trechos para o manual. O cliente, na
verdade, pergunta sobre **prazos** (reembolso e garantia). Esse é o caso típico em que a reordenação com o LLM, que
lê a pergunta inteira, ajuda.

## 7. Com e sem reordenação

`rerank_with_llm` (módulo 02) manda os candidatos para o LLM dar uma nota de 0 a 3 e reordena. Cada pergunta custa
**uma chamada** ao modelo com todos os candidatos (até 24 na híbrida, 700 caracteres de cada), então rodamos em um
subconjunto pequeno: seis perguntas difíceis para a híbrida (rr < 0,5), duas medianas e duas que ela já acerta, para
conferir que a reordenação não piora o que está bom.

In [15]:
subset_ids = ["q01", "q02", "q05", "q08", "q09", "q11", "q15", "q21", "q23", "q25"]
subset = [questions_by_id[i] for i in subset_ids]

rows, rerank_tables = [], {}
for rerank in (False, True):
    label = "hybrid k=4 + rerank" if rerank else "hybrid k=4"
    t, s = evaluate_retrieval(lambda q: retriever.search(q, k=4, mode="hybrid", rerank=rerank), subset)
    rerank_tables[label] = t
    rows.append({"config": label, **s})

pd.DataFrame(rows).set_index("config")

2026-09-27 22:17:28 | INFO     | src.models | Inicializando LLM | provedor=azure | deployment=gpt-5.4-mini


,questions,hit_rate,mrr,coverage
config,,,,
hybrid k=4,10,0.8,0.425,0.75
hybrid k=4 + rerank,10,1.0,0.600,1.00


In [16]:
side_by_side = rerank_tables["hybrid k=4"][["id", "type", "rr", "coverage"]].merge(
    rerank_tables["hybrid k=4 + rerank"][["id", "rr", "coverage", "retrieved"]],
    on="id",
    suffixes=("", "_rerank"),
)
side_by_side

,id,type,rr,coverage,rr_rerank,coverage_rerank,retrieved
0,q01,fact,0.500000,1.0,0.5,1.0,"[faq.csv, policies/shipping.md, policies/shipping.md, policies/cancellation.md]"
1,q02,fact,0.000000,0.0,0.5,1.0,"[help_center/warranty_claim_step_by_step.html, policies/warranty.md, policies/warranty..."
2,q05,fact,0.500000,1.0,0.5,1.0,"[faq.csv, manuals/fn-001_nova_buds.pdf, manuals/fn-001_nova_buds.pdf, manuals/fn-001_n..."
3,q08,fact,1.000000,1.0,1.0,1.0,"[help_center/nova_pontos_loyalty_program.html, policies/payment.md, faq.csv, help_cent..."
4,q09,fact,0.250000,1.0,0.5,1.0,"[help_center/change_delivery_address.html, help_center/support_channels_and_hours.html..."
5,q11,fact,1.000000,1.0,1.0,1.0,"[help_center/coupon_not_applied.html, help_center/coupon_not_applied.html, help_center..."
6,q15,procedure,0.333333,1.0,0.5,1.0,"[manuals/fn-001_nova_buds.pdf, help_center/warranty_claim_step_by_step.html, help_cent..."
7,q21,multi_document,0.000000,0.0,0.5,1.0,"[help_center/warranty_claim_step_by_step.html, policies/refund.md, policies/warranty.m..."
8,q23,multi_document,0.333333,0.5,0.5,1.0,"[faq.csv, manuals/cx-001_nova_boom.pdf, manuals/cx-001_nova_boom.pdf, policies/warrant..."
9,q25,multi_document,0.333333,1.0,0.5,1.0,"[faq.csv, policies/payment.md, policies/shipping.md, help_center/business_purchases.html]"


**Observe** (na nossa execução, no subconjunto: hit rate 0,8 → 1,0; MRR 0,425 → 0,600; coverage 0,75 → 1,0):

- a reordenação sobe a fonte certa para o topo e, principalmente, **completa** as `multi_document`: na `q21`, as
  políticas de reembolso e garantia entram na lista, porque o LLM entende que a pergunta é sobre prazos;
- nas perguntas que já estavam boas, nada piora;
- o ganho parece enorme porque **escolhemos** perguntas difíceis. Em todo o conjunto a diferença seria menor. Ao
  comparar configurações, use o mesmo conjunto para as duas (aqui, o subconjunto nos dois lados);
- o preço é uma chamada ao LLM por pergunta, com milhares de tokens de entrada, e mais latência. O produto deixa
  `rerank=False` por padrão, e o grafo corretivo da aula 3.2 já faz um filtro parecido no nó `grade`. Na aula 3.5
  voltamos a esse custo: reordenar só quando for preciso.

## 8. Do notebook para o projeto

### [src/evaluation.py](../src/evaluation.py), seção 1: Recuperação

| Função | O que faz |
|---|---|
| `load_questions(path=None)` | lê `data/eval/questions.json` (ou outro arquivo, como o do projeto final) |
| `retrieval_metrics(retrieved, expected)` | `hit`, `rr` e `coverage` de uma pergunta, a partir das **fontes** |
| `evaluate_retrieval(search, questions=None)` | roda a busca em cada pergunta **com fonte esperada** e devolve `(tabela, resumo)` |

Decisões de projeto:

- **a busca entra como função**: o módulo de avaliação não conhece o `HybridRetriever`, então avalia qualquer
  retriever, qualquer configuração, ou um serviço externo;
- **as `no_answer` são filtradas pela lista vazia** de `expected_sources`, não pelo tipo: uma pergunta nova sem
  fonte esperada também fica de fora automaticamente;
- **a tabela guarda `retrieved` e `expected`**: é ela que permite a análise de erros, não só a média;
- **o resumo é um dicionário simples**, fácil de registrar e comparar entre versões.

Como `HybridRetriever` é um `BaseRetriever`, o próprio `.invoke` já é uma função de busca (com a configuração
padrão do produto):

In [17]:
import inspect

from src import evaluation

print(inspect.getsource(evaluation.retrieval_metrics))

_, product_summary = evaluate_retrieval(retriever.invoke)
print("Configuração padrão do produto:", product_summary)
print(f"Chamadas à API de embeddings na aula inteira: {embeddings.api_calls}")

def retrieval_metrics(retrieved: list[str], expected: list[str]) -> dict:
    expected_set = set(expected)
    positions = [i for i, source in enumerate(retrieved, start=1) if source in expected_set]
    return {
        "hit": int(bool(positions)),
        "rr": 1 / positions[0] if positions else 0.0,
        "coverage": len(expected_set & set(retrieved)) / len(expected_set) if expected_set else 0.0,
    }

Configuração padrão do produto: {'questions': 26, 'hit_rate': 0.923, 'mrr': 0.683, 'coverage': 0.904}
Chamadas à API de embeddings na aula inteira: 26


> 💡 **Dica:** guarde o resumo da configuração do produto junto com a versão do índice (`index_info()`) sempre que
> mudar algo na ingestão ou na busca. No projeto final, é exatamente essa comparação "antes x depois" que vai dizer
> se os documentos novos pioraram alguma pergunta antiga.

## Resumo

- Um **conjunto de avaliação fixo** (pergunta, tipo, resposta de referência, fontes esperadas) transforma opinião
  em número e permite comparar versões.
- As fontes esperadas são **arquivos**, não trechos, para o gabarito sobreviver a mudanças de chunking.
- Perguntas `no_answer` ficam fora das métricas de busca: não há trecho certo a encontrar; elas avaliam a geração.
- **hit@k** (chegou?), **reciprocal rank / MRR** (em que posição?) e **coverage** (chegaram todas as fontes?) são
  determinísticas e baratas.
- Nesta base, com `k = 4`: vetorial > híbrida > BM25 em hit rate e MRR; `k` maior aumenta hit e coverage.
- Analise os erros pergunta a pergunta: parte deles é **gabarito incompleto** (a FAQ também responde), parte é
  foco errado, documentos vizinhos ou vocabulário diferente.
- A reordenação com o LLM melhora as perguntas difíceis, mas custa uma chamada por pergunta.

## Próxima aula

Na aula 3.5 avaliamos a **resposta**: rodamos o RAG completo em perguntas de todos os tipos (inclusive as
`no_answer`) e usamos um LLM como juiz para medir alucinação, correção e completude.